In [23]:
import os
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
from opendartreader import OpenDartReader

# API 키 로드 및 DART 객체 생성
load_dotenv()
api_key = os.environ.get('DART_API_KEY')
dart = OpenDartReader(api_key)

# 매핑 테이블 불러오기
current_path = os.getcwd() # 현재 위치: .../investment-risk-detector/notebooks
root_path = os.path.dirname(current_path) # 한 단계 위(프로젝트 루트)로 이동: .../investment-risk-detector
csv_path = os.path.join(root_path, 'data', 'kospi_top50_mapping.csv') # 루트/data/... 로 경로 재지정

print(f"✅ 파일을 불러올 경로: {csv_path}")

# CSV에서 숫자를 읽어올 때 앞의 '0'이 날아갈 수 있으므로 문자열(str)
mapping_df = pd.read_csv(csv_path, dtype={'corp_code': str}) 

# 삼성전자 고유번호 추출 및 8자리 포맷팅(zfill)
raw_code = mapping_df.loc[mapping_df['corp_name'] == '삼성전자', 'corp_code'].values[0]
samsung_corp_code = str(raw_code).zfill(8) # '126380' -> '00126380' 변환
print(f"✅ 삼성전자 DART 고유번호: {samsung_corp_code}")

# 조회 기간 설정 (오늘 기준 1년 전)
today = datetime.now()
one_year_ago = today - relativedelta(years=1)

# API가 요구하는 'YYYYMMDD' 포맷으로 변환
bgn_de = one_year_ago.strftime('%Y%m%d')
end_de = today.strftime('%Y%m%d')
print(f"✅ 조회 기간: {bgn_de} ~ {end_de}")

# OpenDART API로 공시 목록 조회
samsung_disclosures = dart.list(samsung_corp_code, start=bgn_de, end=end_de)

# 결과 확인
print(f"✅ 수집된 공시 개수: {len(samsung_disclosures)}건")
display(samsung_disclosures.head(15))

✅ 파일을 불러올 경로: C:\Users\YUN\Project\investment-risk-detector\data\kospi_top50_mapping.csv
✅ 삼성전자 DART 고유번호: 00126380
✅ 조회 기간: 20250828 ~ 20260828
✅ 수집된 공시 개수: 2847건


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260826000445,김경륜,20260826,
1,00126380,삼성전자,005930,Y,[기재정정]임원ㆍ주요주주특정증권등소유상황보고서,20260826000434,조미선,20260826,
2,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260824000222,안성준,20260824,
3,00126380,삼성전자,005930,Y,기타경영사항(자율공시),20260821800837,삼성전자,20260821,유
4,00126380,삼성전자,005930,Y,주요사항보고서(자기주식취득결정),20260821000616,삼성전자,20260821,
5,00126380,삼성전자,005930,Y,수시공시의무관련사항(공정공시),20260821800763,삼성전자,20260821,유
6,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260821000262,이명재,20260821,
7,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260820000290,김경태,20260820,
8,00126380,삼성전자,005930,Y,임원ㆍ주요주주특정증권등소유상황보고서,20260820000040,이희곤,20260820,
9,00126380,삼성전자,005930,Y,[기재정정]임원ㆍ주요주주특정증권등소유상황보고서,20260814003973,박태훈,20260814,


In [24]:
# 제출인명(flr_nm)이 법인명('삼성전자')과 일치하는 데이터만 필터링
corp_name = '삼성전자'
filtered_df = samsung_disclosures[samsung_disclosures['flr_nm'] == corp_name].copy()

# 필터링 전후 데이터 건수 비교
original_count = len(samsung_disclosures)
filtered_count = len(filtered_df)
print(f"✅ 전체 공시 건수: {original_count}건")
print(f"✅ 법인 제출 공시 건수: {filtered_count}건 (노이즈 {original_count - filtered_count}건 제거됨!)")

# 결과 확인
display(filtered_df[['rcept_dt', 'report_nm', 'flr_nm']].head(15))

✅ 전체 공시 건수: 2847건
✅ 법인 제출 공시 건수: 120건 (노이즈 2727건 제거됨!)


,rcept_dt,report_nm,flr_nm
3,20260821,기타경영사항(자율공시),삼성전자
4,20260821,주요사항보고서(자기주식취득결정),삼성전자
5,20260821,수시공시의무관련사항(공정공시),삼성전자
12,20260814,반기보고서 (2026.06),삼성전자
14,20260814,동일인등출자계열회사와의상품ㆍ용역거래변경,삼성전자
15,20260814,동일인등출자계열회사와의상품ㆍ용역거래변경,삼성전자
16,20260814,동일인등출자계열회사와의상품ㆍ용역거래변경,삼성전자
17,20260814,지급수단별ㆍ지급기간별지급금액및분쟁조정기구에관한사항,삼성전자
43,20260731,최대주주등소유주식변동신고서,삼성전자
47,20260730,특수관계인과의보험거래,삼성전자


In [25]:
# 리스크 탐지 키워드 리스트 정의
risk_keywords = [
    '유상증자', '감자', '소송', '횡령', '배임', 
    '해지', '생산중단', '영업정지', '전환사채', '신주인수권부사채'
]

# 리스트를 정규표현식 OR(|) 패턴으로 변환 (예: '유상증자|감자|소송|...')
pattern = '|'.join(risk_keywords)
print(f"✅ 탐지 패턴: {pattern}")

# report_nm(보고서명)에 키워드가 포함된 공시만 추출
risk_df = filtered_df[filtered_df['report_nm'].str.contains(pattern, regex=True, na=False)].copy()

# 결과 확인
print(f"✅ 1년간 발견된 잠재적 위험 공시 건수: {len(risk_df)}건")
display(risk_df[['rcept_dt', 'report_nm', 'rcept_no']])

✅ 탐지 패턴: 유상증자|감자|소송|횡령|배임|해지|생산중단|영업정지|전환사채|신주인수권부사채
✅ 1년간 발견된 잠재적 위험 공시 건수: 0건


,rcept_dt,report_nm,rcept_no


In [26]:
import time

# 파이프라인 시작
all_risk_disclosures = [] # 필터링된 공시들을 담을 빈 리스트

print("🚀 KOSPI 상위 50개 기업 리스크 공시 수집 시작...")

for index, row in mapping_df.iterrows():
    corp_name = row['corp_name']
    corp_code = str(row['corp_code']).zfill(8)
    
    print(f"[{index+1}/50] {corp_name}({corp_code}) 분석 중...")
    
    try:
        # 공시 목록 수집
        disclosures = dart.list(corp_code, start=bgn_de, end=end_de)
        
        # 공시가 아예 없는 경우(결과가 빈 데이터프레임이거나 None일 때) 예외 처리
        if disclosures is None or disclosures.empty:
            continue
            
        # 필터링 1: 제출인명(flr_nm)이 법인명과 일치하는 공시만 남기기
        filtered_df = disclosures[disclosures['flr_nm'] == corp_name].copy()
        
        # 필터링 2: 보고서명(report_nm)에 위험 키워드가 포함된 공시만 추출[cite: 1]
        risk_df = filtered_df[filtered_df['report_nm'].str.contains(pattern, regex=True, na=False)].copy()
        
        # 발견된 위험 공시가 있다면 리스트에 추가
        if not risk_df.empty:
            all_risk_disclosures.append(risk_df)
            
    except Exception as e:
        print(f"❌ {corp_name} 수집 중 에러 발생: {e}")
        
    # API 호출 제한 방지 (0.5초 대기)
    time.sleep(0.5) 

# 수집된 결과 병합 및 저장
if all_risk_disclosures:
    # 리스트에 담긴 여러 데이터프레임을 하나로 합침
    final_risk_df = pd.concat(all_risk_disclosures, ignore_index=True)
    
    # 결과를 새로운 CSV로 저장
    save_path = os.path.join(root_path, 'data', 'kospi50_risk_disclosures.csv')
    final_risk_df.to_csv(save_path, index=False, encoding='utf-8-sig')
    
    print(f"\n✅ 파이프라인 수집 완료! 총 {len(final_risk_df)}건의 위험 공시가 저장되었습니다.")
    display(final_risk_df[['corp_name', 'rcept_dt', 'report_nm']].head(15))
else:
    print("\n✅ 수집 완료! (해당 기간 내 KOSPI 50 기업의 룰베이스 위험 공시가 없습니다.)")

🚀 KOSPI 상위 50개 기업 리스크 공시 수집 시작...
[1/50] 삼성전자(00126380) 분석 중...
[2/50] SK하이닉스(00164779) 분석 중...
[3/50] SK스퀘어(01596425) 분석 중...
[4/50] 삼성전기(00126371) 분석 중...
[5/50] 현대차(00164742) 분석 중...
[6/50] LG에너지솔루션(01515323) 분석 중...
[7/50] 삼성바이오로직스(00877059) 분석 중...
[8/50] 삼성물산(00149655) 분석 중...
[9/50] 삼성생명(00126256) 분석 중...
[10/50] KB금융(00688996) 분석 중...
[11/50] 한화에어로스페이스(00126566) 분석 중...
[12/50] 두산에너빌리티(00159616) 분석 중...
[13/50] 기아(00106641) 분석 중...
[14/50] 신한지주(00382199) 분석 중...
[15/50] HD현대중공업(01390344) 분석 중...
[16/50] 셀트리온(00413046) 분석 중...
[17/50] 현대모비스(00164788) 분석 중...
[18/50] 삼성SDI(00126362) 분석 중...
[19/50] SK(00181712) 분석 중...
[20/50] 하나금융지주(00547583) 분석 중...
[21/50] NAVER(00266961) 분석 중...
[22/50] LG전자(00401731) 분석 중...
[23/50] 삼성화재(00139214) 분석 중...
[24/50] LS ELECTRIC(00105855) 분석 중...
[25/50] 고려아연(00102858) 분석 중...
[26/50] 한화오션(00111704) 분석 중...
[27/50] HD현대일렉트릭(01205851) 분석 중...
[28/50] 효성중공업(01316245) 분석 중...
[29/50] POSCO홀딩스(00155319) 분석 중...
[30/50] HD한국조선해양(00164830) 분석 중...
[31

,corp_name,rcept_dt,report_nm
0,SK하이닉스,20260715,유상증자또는주식관련사채등의발행결과(자율공시)
1,SK하이닉스,20260710,[기재정정]주요사항보고서(유상증자결정)
2,SK하이닉스,20260303,[기재정정]유상증자결정(종속회사의주요경영사항)
3,SK하이닉스,20260303,유상증자결정(종속회사의주요경영사항)
4,SK스퀘어,20260710,[기재정정]주요사항보고서(유상증자결정)(자회사의 주요경영사항)
5,SK스퀘어,20251002,신탁계약해지결과보고서
6,SK스퀘어,20250930,주요사항보고서(자기주식취득신탁계약해지결정)
7,LG에너지솔루션,20260225,[기재정정]유상증자결정(종속회사의주요경영사항)
8,LG에너지솔루션,20251226,단일판매ㆍ공급계약해지
9,LG에너지솔루션,20251217,단일판매ㆍ공급계약해지
